In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.svm import SVC  # Using SVM for clear boundary visualization
from sklearn.preprocessing import StandardScaler

# Load the dataset containing original & counterfactual pairs
df = pd.read_csv('counterfactual_pairs_all_with_outcomes.csv')

# Drop non-numeric columns if any
df = df.select_dtypes(include=[np.number])

# Extract original and counterfactual points
originals = df.iloc[::2, :-1].values  # Even indices: original instances
counterfactuals = df.iloc[1::2, :-1].values  # Odd indices: counterfactuals

# Extract class labels
original_labels = df.iloc[::2, -1].values  # Class labels for original instances
counterfactual_labels = df.iloc[1::2, -1].values  # Class labels for counterfactuals

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(np.vstack((originals, counterfactuals)))

# Apply PCA to reduce to 2 components
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# Split PCA transformed data into original and counterfactual
originals_pca = X_pca[:len(originals)]
counterfactuals_pca = X_pca[len(originals):]

# Fit an SVM classifier to visualize decision boundary
y_labels = np.array([0] * len(originals) + [1] * len(counterfactuals))
clf = SVC(kernel='linear')  # Using linear kernel for a clear boundary
clf.fit(X_pca, y_labels)

# Generate mesh grid for decision boundary
x_min, x_max = X_pca[:, 0].min() - 1, X_pca[:, 0].max() + 1
y_min, y_max = X_pca[:, 1].min() - 1, X_pca[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100),
                     np.linspace(y_min, y_max, 100))
Z = clf.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

# Plot decision boundary
plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')

# Plot original and counterfactual data points with class-based colors
plt.scatter(originals_pca[original_labels == 0, 0], originals_pca[original_labels == 0, 1], c='blue', label='Original Class 0', edgecolors='k')
plt.scatter(originals_pca[original_labels == 1, 0], originals_pca[original_labels == 1, 1], c='cyan', label='Original Class 1', edgecolors='k')
plt.scatter(counterfactuals_pca[counterfactual_labels == 0, 0], counterfactuals_pca[counterfactual_labels == 0, 1], c='red', label='Counterfactual Class 0', edgecolors='k')
plt.scatter(counterfactuals_pca[counterfactual_labels == 1, 0], counterfactuals_pca[counterfactual_labels == 1, 1], c='magenta', label='Counterfactual Class 1', edgecolors='k')

# Connect each original-counterfactual pair with a bisector line
for orig, cf in zip(originals_pca, counterfactuals_pca):
    plt.plot([orig[0], cf[0]], [orig[1], cf[1]], 'k--', alpha=0.5)  # Draw bisector line

plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('Decision Boundary with Original-Counterfactual Pairs')
plt.legend()
plt.show()

# Select only 300 data points with equal instances of 0 and 1
df_0 = df[df['Outcome'] == 0].sample(n=20, random_state=42)
df_1 = df[df['Outcome'] == 1].sample(n=20, random_state=42)
df_balanced = pd.concat([df_0, df_1])

# Extract original and counterfactual points from the balanced dataset
originals_balanced = df_balanced.iloc[::2, :-1].values  # Even indices: original instances
counterfactuals_balanced = df_balanced.iloc[1::2, :-1].values  # Odd indices: counterfactuals

# Extract class labels from the balanced dataset
original_labels_balanced = df_balanced.iloc[::2, -1].values  # Class labels for original instances
counterfactual_labels_balanced = df_balanced.iloc[1::2, -1].values  # Class labels for counterfactuals

# Standardize features
X_scaled_balanced = scaler.fit_transform(np.vstack((originals_balanced, counterfactuals_balanced)))

# Apply PCA to reduce to 2 components
X_pca_balanced = pca.fit_transform(X_scaled_balanced)

# Split PCA transformed data into original and counterfactual
originals_pca_balanced = X_pca_balanced[:len(originals_balanced)]
counterfactuals_pca_balanced = X_pca_balanced[len(originals_balanced):]

# Fit an SVM classifier to visualize decision boundary
y_labels_balanced = np.array([0] * len(originals_balanced) + [1] * len(counterfactuals_balanced))
clf_balanced = SVC(kernel='linear')  # Using linear kernel for a clear boundary
clf_balanced.fit(X_pca_balanced, y_labels_balanced)

# Generate mesh grid for decision boundary
x_min_balanced, x_max_balanced = X_pca_balanced[:, 0].min() - 1, X_pca_balanced[:, 0].max() + 1
y_min_balanced, y_max_balanced = X_pca_balanced[:, 1].min() - 1, X_pca_balanced[:, 1].max() + 1
xx_balanced, yy_balanced = np.meshgrid(np.linspace(x_min_balanced, x_max_balanced, 100),
                                       np.linspace(y_min_balanced, y_max_balanced, 100))
Z_balanced = clf_balanced.predict(np.c_[xx_balanced.ravel(), yy_balanced.ravel()])
Z_balanced = Z_balanced.reshape(xx_balanced.shape)

# Plot decision boundary
plt.figure(figsize=(8, 6))
plt.contourf(xx_balanced, yy_balanced, Z_balanced, alpha=0.3, cmap='coolwarm')

# Plot original and counterfactual data points with class-based colors
plt.scatter(originals_pca_balanced[original_labels_balanced == 0, 0], originals_pca_balanced[original_labels_balanced == 0, 1], c='blue', label='Original Class 0', edgecolors='k')
plt.scatter(originals_pca_balanced[original_labels_balanced == 1, 0], originals_pca_balanced[original_labels_balanced == 1, 1], c='cyan', label='Original Class 1', edgecolors='k')
plt.scatter(counterfactuals_pca_balanced[counterfactual_labels_balanced == 0, 0], counterfactuals_pca_balanced[counterfactual_labels_balanced == 0, 1], c='red', label='Counterfactual Class 0', edgecolors='k')
plt.scatter(counterfactuals_pca_balanced[counterfactual_labels_balanced == 1, 0], counterfactuals_pca_balanced[counterfactual_labels_balanced == 1, 1], c='magenta', label='Counterfactual Class 1', edgecolors='k')

# Connect each original-counterfactual pair with a bisector line
for orig, cf in zip(originals_pca_balanced, counterfactuals_pca_balanced):
    plt.plot([orig[0], cf[0]], [orig[1], cf[1]], 'k--', alpha=0.5)  # Draw bisector line

plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('Decision Boundary with Original-Counterfactual Pairs (Balanced Data)')
plt.legend()
plt.show()